# Prefix-Conditioned Rollout Exploration (PCRE)

## Motivation

Standard GRPO assigns reward to the **full completion**. When a rollout is wrong, every token — including perfectly reasonable reasoning steps — gets penalised. Conversely, GRPO only reinforces the exact winning trace, so the model can memorise a single pattern rather than learning to reason robustly.

```
Full rollout:  [a1 a2 a3 a4 a5 a6 → wrong_answer]   → penalise a1…a6
               ↑ but [a1 a2 a3 a4] might be a fine prefix!
```

## The Idea

Take past rollouts (correct **or** incorrect), randomly truncate to a prefix, and have the model **continue from there**. The reward is applied only to the generated suffix.

```
Past rollout:  [a1 a2 a3 a4 a5 a6 → wrong]
                      ↓  truncate at 60%
Prefix prompt: Q + assistant: [a1 a2 a3]
               ↓  model generates
New suffix:    [a4' a5' → correct_answer]   → reward = 1.0
```

**Goal**: the model must find the right answer from any reasoning state — it cannot rely on a memorised winning prefix.

---

This notebook walks through:
1. The `PrefixRolloutBuffer` mechanics
2. Prefix truncation and prompt formatting
3. How `PrefixAugmentedDataset` modifies training examples
4. The reward signal on prefix-augmented completions
5. End-to-end data flow

In [1]:
import os, re, sys, random
import textwrap
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from datasets import load_dataset
from transformers import AutoTokenizer

sys.path.insert(0, os.path.abspath('.'))  # make 'src' importable
from src.prefix_rollout import (
    PrefixRolloutBuffer,
    PrefixRolloutCollector,
    PrefixAugmentedDataset,
)

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Tokenizer loaded: Qwen/Qwen3-0.6B


In [2]:
# Load GSM8K and format for GRPO
dataset = load_dataset("openai/gsm8k", "main")

def extract_gold_answer(text):
    m = re.search(r"####\s*(.+)", text)
    return m.group(1).strip().replace(",", "") if m else ""

def format_example(ex):
    ex["prompt"] = [{"role": "user", "content": ex["question"]}]
    ex["gold_answer"] = extract_gold_answer(ex["answer"])
    return ex

train_dataset = dataset["train"].map(format_example)
print(f"Train: {len(train_dataset)} examples")
print(f"\nExample question: {train_dataset[0]['question'][:120]}...")
print(f"Gold answer: {train_dataset[0]['gold_answer']}")

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Train: 7473 examples

Example question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natali...
Gold answer: 72


---
## Part 1 — The Rollout Buffer

The `PrefixRolloutBuffer` is a FIFO circular buffer that stores `(question, completion_text, is_correct)` triples.  
It's filled by the `PrefixRolloutCollector` reward function, which runs as a side-effect during GRPO's reward evaluation — returning `0.0` for all rollouts so it has no training signal of its own.

In [3]:
# --- Craft some mock rollouts (mimicking what a model might generate) ---

MOCK_ROLLOUTS = [
    (
        "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. "
        "How many clips did Natalia sell altogether in April and May?",
        # CORRECT rollout
        "Let me think step by step.\n"
        "In April, Natalia sold 48 clips.\n"
        "In May, she sold half as many: 48 / 2 = 24 clips.\n"
        "Total: 48 + 24 = 72 clips.\n"
        "#### 72",
        True,
    ),
    (
        "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. "
        "How many clips did Natalia sell altogether in April and May?",
        # INCORRECT rollout (arithmetic mistake)
        "Natalia sold 48 clips in April.\n"
        "Half of 48 is 24, so she sold 24 more.\n"
        "Wait, half as many means she sold fewer.\n"
        "48 - 24 = 24 total clips.\n"
        "#### 24",
        False,
    ),
    (
        "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. "
        "How much did she earn?",
        # CORRECT
        "Weng earns $12 per hour.\n"
        "50 minutes = 50/60 hours.\n"
        "Earnings = 12 × (50/60) = 12 × 5/6 = 10.\n"
        "#### 10",
        True,
    ),
    (
        "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. "
        "Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. "
        "How much more money does Betty need to buy the wallet?",
        # INCORRECT
        "Betty needs $100.\n"
        "She has half: $50.\n"
        "Parents give $15. Grandparents give $15 × 2 = $30.\n"
        "She now has: 50 + 15 + 30 = $90.\n"
        "She still needs: 100 - 90 = $10.\n"
        "#### 10",
        True,  # this one happens to be correct!
    ),
]

buf = PrefixRolloutBuffer(max_size=100, min_prefix_frac=0.15, max_prefix_frac=0.75)

for question, completion, is_correct in MOCK_ROLLOUTS:
    buf.add(question, completion, is_correct)

print("Buffer stats:", buf.get_stats())

Buffer stats: {'buffer_size': 4, 'n_correct': 3, 'n_incorrect': 1}


---
## Part 2 — Prefix Truncation

When an example is sampled for augmentation, the stored completion is truncated at a random position between `[min_prefix_frac, max_prefix_frac]` of its total character length.

Let's visualise what the prefix looks like at different truncation fractions.

In [4]:
FRACS = [0.15, 0.30, 0.50, 0.75]

question, full_completion, is_correct = MOCK_ROLLOUTS[0]  # correct rollout
L = len(full_completion)

print(f"Question: {question[:80]}...")
print(f"Full completion ({L} chars, correct={is_correct}):")
print(f"  {repr(full_completion)}")
print()
print("=" * 70)
for frac in FRACS:
    cut = int(L * frac)
    prefix = full_completion[:cut]
    print(f"\n── frac={frac:.2f}  cut={cut}  chars remaining={L - cut} ──")
    print(f"  PREFIX : {repr(prefix)}")
    print(f"  SUFFIX : {repr(full_completion[cut:])}")

Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many...
Full completion (144 chars, correct=True):
  'Let me think step by step.\nIn April, Natalia sold 48 clips.\nIn May, she sold half as many: 48 / 2 = 24 clips.\nTotal: 48 + 24 = 72 clips.\n#### 72'


── frac=0.15  cut=21  chars remaining=123 ──
  PREFIX : 'Let me think step by '
  SUFFIX : 'step.\nIn April, Natalia sold 48 clips.\nIn May, she sold half as many: 48 / 2 = 24 clips.\nTotal: 48 + 24 = 72 clips.\n#### 72'

── frac=0.30  cut=43  chars remaining=101 ──
  PREFIX : 'Let me think step by step.\nIn April, Natali'
  SUFFIX : 'a sold 48 clips.\nIn May, she sold half as many: 48 / 2 = 24 clips.\nTotal: 48 + 24 = 72 clips.\n#### 72'

── frac=0.50  cut=72  chars remaining=72 ──
  PREFIX : 'Let me think step by step.\nIn April, Natalia sold 48 clips.\nIn May, she '
  SUFFIX : 'sold half as many: 48 / 2 = 24 clips.\nTotal: 48 + 24 = 72 clips.\n#### 72'

── frac=0.75  cut=108  chars remaining=36 ─

---
## Part 3 — Prefix Prompt Formatting

The prefix is injected into the prompt using `tokenizer.apply_chat_template(..., continue_final_message=True)`.  
This produces a raw string that ends **without** `<|im_end|>` — so when TRL feeds this to the model, generation naturally continues from the end of the prefix.

Compare:
- **Standard prompt**: `<|im_start|>user\n{Q}<|im_end|>\n<|im_start|>assistant\n`  
- **Prefix prompt**: `<|im_start|>user\n{Q}<|im_end|>\n<|im_start|>assistant\n{prefix}` ← no closing token

In [5]:
def show_prompt(label, text, max_chars=400):
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(text[:max_chars] + ("..." if len(text) > max_chars else ""))
    print(f"  [ends with: {repr(text[-30:])}]")

# Standard prompt (no prefix)
std_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": question}],
    tokenize=False, add_generation_prompt=True
)
show_prompt("STANDARD PROMPT (model generates full answer)", std_prompt)

# Prefix prompt at 30% truncation
cut = int(L * 0.30)
prefix_30 = full_completion[:cut]
prefix_prompt_30 = tokenizer.apply_chat_template(
    [{"role": "user", "content": question},
     {"role": "assistant", "content": prefix_30}],
    tokenize=False, add_generation_prompt=False, continue_final_message=True
)
show_prompt(f"PREFIX PROMPT @ 30% (model continues from mid-reasoning)", prefix_prompt_30)

# Prefix prompt at 60% truncation
cut2 = int(L * 0.60)
prefix_60 = full_completion[:cut2]
prefix_prompt_60 = tokenizer.apply_chat_template(
    [{"role": "user", "content": question},
     {"role": "assistant", "content": prefix_60}],
    tokenize=False, add_generation_prompt=False, continue_final_message=True
)
show_prompt(f"PREFIX PROMPT @ 60% (model continues just before answer)", prefix_prompt_60)


────────────────────────────────────────────────────────────
  STANDARD PROMPT (model generates full answer)
────────────────────────────────────────────────────────────
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant

  [ends with: 'm_end|>\n<|im_start|>assistant\n']

────────────────────────────────────────────────────────────
  PREFIX PROMPT @ 30% (model continues from mid-reasoning)
────────────────────────────────────────────────────────────
<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
<think>

</think>

Let me think step by step.
In April, Natali
  [ends with: 'step by step.\nIn April, Natali']

────────────────────────────────────────────────────────────
  PREFIX PRO

In [6]:
# Token count comparison
std_ids = tokenizer(std_prompt, add_special_tokens=False)["input_ids"]
p30_ids = tokenizer(prefix_prompt_30, add_special_tokens=False)["input_ids"]
p60_ids = tokenizer(prefix_prompt_60, add_special_tokens=False)["input_ids"]

print(f"Standard prompt tokens : {len(std_ids)}")
print(f"Prefix @30% prompt tokens: {len(p30_ids)}  (+{len(p30_ids)-len(std_ids)} prefix tokens)")
print(f"Prefix @60% prompt tokens: {len(p60_ids)}  (+{len(p60_ids)-len(std_ids)} prefix tokens)")
print()
print("With max_completion_length=512:")
for label, n in [("Standard", len(std_ids)), ("Prefix@30%", len(p30_ids)), ("Prefix@60%", len(p60_ids))]:
    remaining = 512 - (n - len(std_ids))
    print(f"  {label:15s}: {remaining} tokens left for model to generate")

Standard prompt tokens : 47
Prefix @30% prompt tokens: 63  (+16 prefix tokens)
Prefix @60% prompt tokens: 77  (+30 prefix tokens)

With max_completion_length=512:
  Standard       : 512 tokens left for model to generate
  Prefix@30%     : 496 tokens left for model to generate
  Prefix@60%     : 482 tokens left for model to generate


---
## Part 4 — PrefixAugmentedDataset in Action

The `PrefixAugmentedDataset` wraps the base HF dataset.  
On each `__getitem__`, with probability `augment_prob`, it samples from the buffer and returns a prefix-augmented item.

Key observations:
- `gold_answer` is **preserved** from the original item — reward functions can still check correctness
- `prompt` becomes a **raw string** (not a message list) — TRL's non-conversational tokenisation path is used
- The buffer fills up during training; `min_buffer=16` prevents augmentation before enough rollouts exist

In [7]:
aug_dataset = PrefixAugmentedDataset(
    train_dataset,
    buf,
    tokenizer,
    augment_prob=0.5,       # 50% augmentation for demo visibility
    from_correct=None,      # sample from all rollouts
    min_buffer=2,           # low threshold so demo triggers
)

print(repr(aug_dataset))
print()

random.seed(0)
for i in range(6):
    item = aug_dataset[i]
    prompt = item["prompt"]
    is_aug = isinstance(prompt, str)  # str = prefix-augmented, list = standard
    gold = item["gold_answer"]
    print(f"Example {i} | augmented={is_aug} | gold_answer={gold}")
    if is_aug:
        # Show where the prefix ends (model continues from here)
        # The prompt ends without <|im_end|>
        tail = prompt[-80:].replace("\n", "↵")
        print(f"  prompt tail: ...{tail!r}")
    else:
        print(f"  prompt: {prompt}")
    print()

PrefixAugmentedDataset(n=7473, augment_prob=0.5, from_correct=None, buffer={'buffer_size': 4, 'n_correct': 3, 'n_incorrect': 1})

Example 0 | augmented=False | gold_answer=72
  prompt: [{'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}]

Example 1 | augmented=False | gold_answer=10
  prompt: [{'content': 'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?', 'role': 'user'}]

Example 2 | augmented=True | gold_answer=5
  prompt tail: ...'art|>assistant↵<think>↵↵</think>↵↵Weng earns $12 per hour.↵50 minutes = 50/60 ho'

Example 3 | augmented=True | gold_answer=42
  prompt tail: ...'start|>assistant↵<think>↵↵</think>↵↵Weng earns $12 per hour.↵50 minutes = 50/60 '

Example 4 | augmented=True | gold_answer=624
  prompt tail: ...'ia sold 48 clips in April.↵Half of 48 is 24, so she sold 24 more.↵W

---
## Part 5 — Reward Signal on Prefix-Augmented Completions

After the model generates a suffix, the **reward is computed on the generated suffix alone** — not on the full `prefix + suffix`. This means:

- The model must produce the `#### <answer>` marker in the **new completion**
- `gold_answer` is preserved from the original dataset item, so correctness checking is unchanged
- The gradient only touches the suffix tokens (the prefix was part of the prompt — no grad flows there)

Let's simulate mock suffix completions to illustrate the reward signal.

In [8]:
def extract_answer(text):
    m = re.search(r"####\s*([\d,\.\-]+)", text)
    if m:
        return m.group(1).strip().replace(",", "")
    nums = re.findall(r"-?[\d,]+\.?\d*", text)
    return nums[-1].replace(",", "") if nums else ""

def correctness_reward(completions, gold_answer, **kwargs):
    rewards = []
    for comp, gold in zip(completions, gold_answer):
        text = comp[0]["content"]
        predicted = extract_answer(text)
        try:
            correct = float(predicted) == float(gold)
        except (ValueError, TypeError):
            correct = False
        rewards.append(1.0 if correct else 0.0)
    return rewards

# ── Scenario: Prefix @ 30% + three possible suffix completions ──
cut = int(L * 0.30)
prefix_30 = full_completion[:cut]

print(f"QUESTION : {question}")
print(f"GOLD     : 72")
print(f"\nPREFIX (what model receives, {cut}/{L} chars):")
print(f"  {repr(prefix_30)}")
print("\n" + "=" * 70)

# Three possible suffixes the model might generate
SUFFIX_SCENARIOS = [
    ("Model completes correctly",
     "In May, she sold half as many: 48 / 2 = 24 clips.\nTotal: 48 + 24 = 72.\n#### 72"),
    ("Model completes incorrectly (arithmetic mistake)",
     "In May, she sold half: 48 / 2 = 24.\nTotal: 48 - 24 = 24.\n#### 24"),
    ("Model generates answer without full reasoning",
     "Total is 72.\n#### 72"),
]

for label, suffix in SUFFIX_SCENARIOS:
    completions = [[{"role": "assistant", "content": suffix}]]
    rewards = correctness_reward(completions, gold_answer=["72"])
    print(f"\n── {label} ──")
    print(f"  Suffix    : {repr(suffix)}")
    print(f"  Reward    : {rewards[0]}   ({'✓ correct' if rewards[0] > 0 else '✗ wrong'})")

QUESTION : Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
GOLD     : 72

PREFIX (what model receives, 43/144 chars):
  'Let me think step by step.\nIn April, Natali'


── Model completes correctly ──
  Suffix    : 'In May, she sold half as many: 48 / 2 = 24 clips.\nTotal: 48 + 24 = 72.\n#### 72'
  Reward    : 1.0   (✓ correct)

── Model completes incorrectly (arithmetic mistake) ──
  Suffix    : 'In May, she sold half: 48 / 2 = 24.\nTotal: 48 - 24 = 24.\n#### 24'
  Reward    : 0.0   (✗ wrong)

── Model generates answer without full reasoning ──
  Suffix    : 'Total is 72.\n#### 72'
  Reward    : 1.0   (✓ correct)


---
## Part 6 — PrefixRolloutCollector: Buffer Filling as a Side-Effect

The `PrefixRolloutCollector` sits in the `reward_funcs` list as a **zero-reward** function.  
It uses the correctness function to label each rollout, then adds everything to the buffer.

In [9]:
fresh_buf = PrefixRolloutBuffer(max_size=100)
collector = PrefixRolloutCollector(fresh_buf, correctness_fn=correctness_reward)

print(f"Buffer before: {fresh_buf.get_stats()}")

# Simulate one GRPO batch: 4 prompts, each with 2 completions (num_generations=2)
batch_prompts = [
    [{"role": "user", "content": MOCK_ROLLOUTS[0][0]}],   # Q1
    [{"role": "user", "content": MOCK_ROLLOUTS[0][0]}],   # Q1 again
    [{"role": "user", "content": MOCK_ROLLOUTS[2][0]}],   # Q2
    [{"role": "user", "content": MOCK_ROLLOUTS[2][0]}],   # Q2 again
]
batch_completions = [
    [{"role": "assistant", "content": "48 + 24 = 72\n#### 72"}],   # correct
    [{"role": "assistant", "content": "48 - 24 = 24\n#### 24"}],   # wrong
    [{"role": "assistant", "content": "10 × (50/60) = 10\n#### 10"}],  # correct
    [{"role": "assistant", "content": "12 / 50 = 0.24\n#### 0.24"}],  # wrong
]
batch_gold = ["72", "72", "10", "10"]

# The collector is called exactly like a reward function:
rewards_from_collector = collector(
    prompts=batch_prompts,
    completions=batch_completions,
    gold_answer=batch_gold,
)

print(f"\nCollector reward output: {rewards_from_collector}  ← always 0.0")
print(f"Buffer after : {fresh_buf.get_stats()}")

print("\nBuffer contents:")
for q, comp, ok in fresh_buf._buf:
    print(f"  is_correct={ok} | {comp[:50]!r}...")

Buffer before: {'buffer_size': 0, 'n_correct': 0, 'n_incorrect': 0}

Collector reward output: [0.0, 0.0, 0.0, 0.0]  ← always 0.0
Buffer after : {'buffer_size': 4, 'n_correct': 2, 'n_incorrect': 2}

Buffer contents:
  is_correct=True | '48 + 24 = 72\n#### 72'...
  is_correct=False | '48 - 24 = 24\n#### 24'...
  is_correct=True | '10 × (50/60) = 10\n#### 10'...
  is_correct=False | '12 / 50 = 0.24\n#### 0.24'...


---
## Part 7 — End-to-End Data Flow

Here is the full lifecycle of a prefix-augmented training step:

In [10]:
# Seed for reproducibility
random.seed(42)

print("=" * 70)
print("  END-TO-END DATA FLOW")
print("=" * 70)

# Step 1: original dataset item
base_item = dict(train_dataset[0])
print("\n[STEP 1] Original dataset item")
print(f"  question    : {base_item['question'][:80]}...")
print(f"  gold_answer : {base_item['gold_answer']}")
print(f"  prompt type : {type(base_item['prompt']).__name__} (message list)")

# Step 2: PrefixAugmentedDataset replaces item with prefix-augmented version
aug_item = fresh_buf.sample_prefix_augmented_item(base_item, tokenizer, from_correct=None)
print("\n[STEP 2] After PrefixAugmentedDataset augmentation")
if aug_item:
    print(f"  prompt type : {type(aug_item['prompt']).__name__} (raw string — TRL non-conversational path)")
    print(f"  gold_answer : {aug_item['gold_answer']}  ← preserved")
    tok_count = len(tokenizer(aug_item["prompt"], add_special_tokens=False)["input_ids"])
    print(f"  prompt length: {tok_count} tokens")
    tail = aug_item["prompt"][-60:].replace("\n", "↵")
    print(f"  prompt ends with: ...{tail!r}")
    print(f"  (no <|im_end|> → model continues from here)")
else:
    print("  (buffer too small — standard item returned)")

# Step 3: Model generates suffix
mock_suffix = "24 clips.\nTotal clips sold = 48 + 24 = 72 clips.\n#### 72"
print("\n[STEP 3] Model generates suffix (mock)")
print(f"  suffix : {repr(mock_suffix)}")

# Step 4: Reward computed on suffix
completions = [[{"role": "assistant", "content": mock_suffix}]]
gold = [base_item["gold_answer"]]
reward = correctness_reward(completions, gold_answer=gold)
print("\n[STEP 4] Reward computed on generated suffix")
print(f"  predicted answer : {extract_answer(mock_suffix)}")
print(f"  gold answer      : {gold[0]}")
print(f"  correctness      : {reward[0]}  ({'✓' if reward[0] > 0 else '✗'})")

# Step 5: Gradient update
print("\n[STEP 5] Gradient update")
print("  → Gradient flows only through SUFFIX tokens")
print("  → PREFIX tokens (in the prompt) receive NO gradient")
print("  → Model learns: 'from this reasoning state, reach the correct answer'")

print("\n" + "=" * 70)

  END-TO-END DATA FLOW

[STEP 1] Original dataset item
  question    : Natalia sold clips to 48 of her friends in April, and then she sold half as many...
  gold_answer : 72
  prompt type : list (message list)

[STEP 2] After PrefixAugmentedDataset augmentation
  prompt type : str (raw string — TRL non-conversational path)
  gold_answer : 72  ← preserved
  prompt length: 54 tokens
  prompt ends with: ...' May?<|im_end|>↵<|im_start|>assistant↵<think>↵↵</think>↵↵48 '
  (no <|im_end|> → model continues from here)

[STEP 3] Model generates suffix (mock)
  suffix : '24 clips.\nTotal clips sold = 48 + 24 = 72 clips.\n#### 72'

[STEP 4] Reward computed on generated suffix
  predicted answer : 72
  gold answer      : 72
  correctness      : 1.0  (✓)

[STEP 5] Gradient update
  → Gradient flows only through SUFFIX tokens
  → PREFIX tokens (in the prompt) receive NO gradient
  → Model learns: 'from this reasoning state, reach the correct answer'



---
## Part 8 — Comparing Prefix Sources: Correct vs Incorrect Rollouts

A key design choice is **which rollouts to sample prefixes from**:

| `--prefix_from_correct` | Effect |
|---|---|
| `correct` | Diversify completions of good reasoning chains (model learns multiple paths to the right answer) |
| `incorrect` | Rescue bad traces — model must recover from a suboptimal start |
| `all` | Both signals simultaneously |

Let's see what prefixes from each pool look like.

In [11]:
# Populate buffer with mixed correct/incorrect rollouts
demo_buf = PrefixRolloutBuffer(max_size=200, min_prefix_frac=0.20, max_prefix_frac=0.70)
for q, comp, ok in MOCK_ROLLOUTS:
    demo_buf.add(q, comp, ok)

print(f"Buffer: {demo_buf.get_stats()}\n")

q_ref = MOCK_ROLLOUTS[0][0]  # same question for all experiments
gold_ref = "72"
base_ref = {"question": q_ref, "prompt": [{"role": "user", "content": q_ref}],
            "gold_answer": gold_ref, "answer": ""}

random.seed(7)
for mode, fc in [("from_correct=True  (correct prefixes)", True),
                 ("from_correct=False (incorrect prefixes)", False),
                 ("from_correct=None  (any rollout)", None)]:
    item = demo_buf.sample_prefix_augmented_item(base_ref, tokenizer, from_correct=fc)
    print(f"── {mode} ──")
    if item:
        # Extract just the assistant content from the prompt string
        # It's after the last <|im_start|>assistant\n
        marker = "<|im_start|>assistant\n"
        idx = item["prompt"].rfind(marker)
        prefix_content = item["prompt"][idx + len(marker):] if idx != -1 else "?"
        print(f"  Prefix: {repr(prefix_content)}")
    else:
        print("  (no candidate found)")
    print()

Buffer: {'buffer_size': 4, 'n_correct': 3, 'n_incorrect': 1}

── from_correct=True  (correct prefixes) ──
  Prefix: '<think>\n\n</think>\n\nWeng earns $12 per hour.\n50 '

── from_correct=False (incorrect prefixes) ──
  Prefix: '<think>\n\n</think>\n\nNatalia sold 48 clips in April.\nHal'

── from_correct=None  (any rollout) ──
  Prefix: '<think>\n\n</think>\n\nLet me think step by step.\nIn April, Natalia sold 48 clips.\nIn May, she sold half as many: 48 / '



---
## Part 9 — How to Use in Training

The full integration into `grpo_gsm8k.py` is already done.  
Use the `--prefix_rollout` flag:

```bash
# Baseline GRPO (no prefix)
python script/grpo_gsm8k.py --max_steps 200

# PCRE — all rollouts, 30% augmentation
python script/grpo_gsm8k.py --max_steps 200 \
    --prefix_rollout \
    --prefix_augment_prob 0.3 \
    --prefix_from_correct all

# PCRE — prefixes only from correct rollouts (diversity within winning traces)
python script/grpo_gsm8k.py --max_steps 200 \
    --prefix_rollout \
    --prefix_augment_prob 0.3 \
    --prefix_from_correct correct

# PCRE — prefixes only from incorrect rollouts (recovery training)
python script/grpo_gsm8k.py --max_steps 200 \
    --prefix_rollout \
    --prefix_augment_prob 0.3 \
    --prefix_from_correct incorrect
```

### Expected effects

- **Robustness**: the model can reach the correct answer from many different reasoning states, not just its favourite one
- **Diversity**: the training signal includes completions from diverse reasoning states, not just standard `[question → answer]`
- **Buffer warmup**: augmentation is suppressed until `min_buffer=16` rollouts exist (~2 GRPO steps), so the first few batches are standard GRPO

### Open questions to explore

1. Does `from_correct=correct` actually improve diversity, or does the model learn to always produce the same suffix regardless of prefix?
2. Does `from_correct=incorrect` help the model recover from bad starts, or does it just add noise?
3. What's the right `augment_prob`? Too high → the model spends most training time continuing rather than starting from scratch; too low → weak signal.
4. Character-level truncation is a rough proxy — token-level truncation at natural sentence boundaries might be more principled.